# Colab: resumable Full-Stack/API adapter training

This notebook trains **one Full-Stack/API specialist adapter** for SQL, REST endpoint contracts, JSON Schema, and structured outputs. It is independent of the other specialist notebooks.

The training mix combines [`b-mc2/sql-create-context`](https://huggingface.co/datasets/b-mc2/sql-create-context) (CC-BY-4.0) with endpoint and schema tasks generated from [GitHub’s official REST OpenAPI description](https://github.com/github/rest-api-description) (MIT). Tasks use structured JSON that can be reviewed or imported into API tools such as Postman. No Kaggle account or API token is needed.

Training uses the pinned `Qwen/Qwen3-0.6B` base and LoRA targets `q_proj`, `gate_proj`, `up_proj`, and `down_proj`, matching the Bridge Q/MLP contract. Google Drive stores resumable checkpoints, logs, telemetry, the final adapter, and a Bridge manifest. Keep the same `RUN_ID` to resume.

In [ ]:
%pip -q install transformers==4.57.6 "peft>=0.15,<1" "datasets>=3.6,<5" accelerate safetensors huggingface_hub tensorboard requests

In [ ]:
import hashlib, json, math, os, random, time
from datetime import datetime, timezone
from pathlib import Path
import requests, torch, transformers
from datasets import Dataset, concatenate_datasets, load_dataset
from huggingface_hub import HfApi
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling, Trainer, TrainerCallback, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from google.colab import drive

if not torch.cuda.is_available(): raise RuntimeError('Select a Colab GPU (T4 or better).')
drive.mount('/content/drive')
RUN_ID='fullstack_api_qwen3_0.6b_q_mlp_r16_v1'
ROOT=Path('/content/drive/MyDrive/cgr_adapter_checkpoints'); OUTPUT_DIR=ROOT/RUN_ID
OUTPUT_DIR.mkdir(parents=True,exist_ok=True); LOG_DIR=OUTPUT_DIR/'tensorboard'; LOG_DIR.mkdir(parents=True,exist_ok=True)
MANIFEST=OUTPUT_DIR/'run_manifest.json'; SPEC_PATH=OUTPUT_DIR/'github_rest_openapi.json'
BASE_MODEL_ID='Qwen/Qwen3-0.6B'; SQL_DATASET_ID='b-mc2/sql-create-context'
API_REPO='github/rest-api-description'; API_FILE='descriptions/api.github.com/api.github.com.json'
TARGET_MODULES=['q_proj','gate_proj','up_proj','down_proj']; RANK,ALPHA=16,32
MAX_LENGTH=1024; MAX_SQL_TRAIN=20000; MAX_SQL_EVAL=512; MAX_API_EVAL=384; API_REPEAT=4
SAVE_STEPS=100; EVAL_STEPS=500; LOGGING_STEPS=10; RESUME=True
api=HfApi(); prior=json.loads(MANIFEST.read_text()) if MANIFEST.exists() else None
if prior:
    if prior.get('base_model_id')!=BASE_MODEL_ID or prior.get('sql_dataset_id')!=SQL_DATASET_ID or prior.get('api_repo')!=API_REPO: raise RuntimeError('RUN_ID has different sources; choose a new RUN_ID.')
    BASE_REV=prior['base_revision']; SQL_REV=prior['sql_revision']; API_COMMIT=prior['api_commit']
else:
    BASE_REV=api.model_info(BASE_MODEL_ID).sha; SQL_REV=api.dataset_info(SQL_DATASET_ID).sha
    r=requests.get(f'https://api.github.com/repos/{API_REPO}/commits/main',timeout=60); r.raise_for_status(); API_COMMIT=r.json()['sha']
if SPEC_PATH.exists() and prior: spec_bytes=SPEC_PATH.read_bytes()
else:
    url=f'https://raw.githubusercontent.com/{API_REPO}/{API_COMMIT}/{API_FILE}'
    r=requests.get(url,timeout=180); r.raise_for_status(); spec_bytes=r.content
    temp=SPEC_PATH.with_suffix('.tmp'); temp.write_bytes(spec_bytes); os.replace(temp,SPEC_PATH)
API_SHA256=hashlib.sha256(spec_bytes).hexdigest()
if prior and prior.get('api_sha256')!=API_SHA256: raise RuntimeError('Saved API spec differs from run manifest; choose a new RUN_ID.')
api_spec=json.loads(spec_bytes); DEVICE=torch.cuda.get_device_name(0); START_UTC=datetime.now(timezone.utc).isoformat()
print('GPU:',DEVICE,'| Transformers:',transformers.__version__)
print('Base:',BASE_REV,'| SQL:',SQL_REV,'| OpenAPI commit:',API_COMMIT)
print('Drive output:',OUTPUT_DIR)

## 1. Prepare SQL and REST/schema tasks

The SQL split is capped at 20,000 train and 512 validation examples by default; set `MAX_SQL_TRAIN = None` to use the complete SQL train split. OpenAPI operations are split before task generation, so examples from one route stay together in training or validation. The API tasks are repeated to keep the smaller spec source represented. Oversized sequences are filtered instead of silently truncated.

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL_ID,revision=BASE_REV)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
SYSTEM='You are a backend and API engineering assistant. Follow the requested output format exactly. For structured tasks return valid JSON only, without Markdown fences or extra prose.'
def chat(user,answer):
    return tokenizer.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':user},{'role':'assistant','content':answer}],tokenize=False,add_generation_prompt=False)

sql=load_dataset(SQL_DATASET_ID,revision=SQL_REV)
if 'train' not in sql: raise RuntimeError(f'No train split: {list(sql.keys())}')
split=sql['train'].train_test_split(test_size=0.03,seed=42); sql_tr,sql_ev=split['train'],split['test']
if MAX_SQL_TRAIN is not None: sql_tr=sql_tr.select(range(min(MAX_SQL_TRAIN,len(sql_tr))))
sql_ev=sql_ev.select(range(min(MAX_SQL_EVAL,len(sql_ev))))
def sql_format(r):
    q=str(r.get('question') or '').strip(); schema=str(r.get('context') or '').strip(); answer=str(r.get('answer') or '').strip()
    if not q or not schema or not answer: return {'text':''}
    prompt='Database schema:\n'+schema+'\n\nQuestion: '+q+'\n\nWrite one SQL query. Return only SQL.'
    return {'text':chat(prompt,answer)}
sql_tr=sql_tr.map(sql_format).filter(lambda r:bool(r['text']))
sql_ev=sql_ev.map(sql_format).filter(lambda r:bool(r['text']))

SCHEMA_KEYS={'$ref','type','format','title','description','enum','const','required','properties','items','additionalProperties','oneOf','anyOf','allOf','nullable','minimum','maximum','minLength','maxLength','pattern','minItems','maxItems','readOnly','writeOnly'}
def compact_schema(s,d=0):
    if not isinstance(s,dict): return s
    if '$ref' in s: return {'$ref':s['$ref']}
    out={}
    for k in SCHEMA_KEYS:
        if k not in s: continue
        v=s[k]
        if k=='properties' and isinstance(v,dict): out[k]={n:compact_schema(x,d+1) for n,x in list(sorted(v.items()))[:24]}
        elif k in {'items','additionalProperties'} and isinstance(v,dict): out[k]=compact_schema(v,d+1)
        elif k in {'oneOf','anyOf','allOf'} and isinstance(v,list): out[k]=[compact_schema(x,d+1) for x in v[:6]]
        elif k in {'required','enum'} and isinstance(v,list): out[k]=v[:32]
        else: out[k]=v
    return {k:out[k] for k in ('$ref','type','format','description') if k in out} if d>=5 else out
def get_json_schema(content):
    for mime,media in sorted((content or {}).items(),key=lambda x:x[0]!='application/json'):
        if 'json' in mime.lower() and isinstance(media,dict) and isinstance(media.get('schema'),dict): return compact_schema(media['schema'])
    return None
def make_op(title,method,path,path_item,op):
    params={}
    for p in list(path_item.get('parameters') or [])+list(op.get('parameters') or []):
        if isinstance(p,dict): params[(p.get('name'),p.get('in'))]={'name':p.get('name'),'in':p.get('in'),'required':bool(p.get('required',False)),'description':str(p.get('description') or '')[:250],'schema':compact_schema(p.get('schema') or {})}
    body=None; b=op.get('requestBody')
    if isinstance(b,dict):
        content={m:compact_schema(v['schema']) for m,v in (b.get('content') or {}).items() if 'json' in m.lower() and isinstance(v,dict) and isinstance(v.get('schema'),dict)}
        if content: body={'required':bool(b.get('required',False)),'description':str(b.get('description') or '')[:250],'content':content}
    responses={}
    for status,res in sorted((op.get('responses') or {}).items()):
        if len(responses)>=4 or not isinstance(res,dict): continue
        row={'description':str(res.get('description') or '')[:250]}; sch=get_json_schema(res.get('content') or {})
        if sch is not None: row['schema']=sch
        responses[status]=row
    return {'api_title':title,'method':method.upper(),'path':path,'operationId':op.get('operationId'),
      'summary':str(op.get('summary') or '')[:400],'description':str(op.get('description') or '')[:500],
      'parameters':list(params.values()),'requestBody':body,'responses':responses}
def api_tasks(op):
    brief={'api_title':op['api_title'],'method':op['method'],'path':op['path'],'operationId':op['operationId'],
      'summary':op['summary'],'description':op['description'],
      'parameters':[{k:p[k] for k in ('name','in','required','description')} for p in op['parameters']],
      'requestBody':None if not op['requestBody'] else {'required':op['requestBody']['required'],'description':op['requestBody']['description'],'content_types':list(op['requestBody']['content'])},
      'responses':{status:row['description'] for status,row in op['responses'].items()}}
    prompt_input=json.dumps(brief,ensure_ascii=False,separators=(',',':'))
    if len(prompt_input)>12000: return []
    contract={k:op[k] for k in ('method','path','operationId','summary','parameters','requestBody','responses')}
    target=json.dumps(contract,ensure_ascii=False,separators=(',',':')); rows=[]
    if len(target)<=10000:
        prompt='Generate a complete OpenAPI endpoint contract from this API requirement. Return JSON with keys method, path, operationId, summary, parameters, requestBody, responses. Include JSON Schema types and required fields where supported. Return valid JSON only.\n\nRequirement:\n'+prompt_input
        rows.append({'text':chat(prompt,target),'kind':'endpoint_contract'})
    if op['requestBody']:
        schema=next(iter(op['requestBody']['content'].values())); target=json.dumps(schema,ensure_ascii=False,separators=(',',':'))
        if len(target)<=10000:
            prompt=f"Generate the JSON Schema for the request body of {op['method']} {op['path']} from its summary, parameter, and body requirements. Return the schema only.\n\nRequirement:\n"+prompt_input
            rows.append({'text':chat(prompt,target),'kind':'request_schema'})
    for status,res in op['responses'].items():
        if status.startswith('2') and isinstance(res.get('schema'),dict):
            target=json.dumps(res['schema'],ensure_ascii=False,separators=(',',':'))
            if len(target)<=10000:
                prompt=f"Generate the JSON response schema for status {status} of {op['method']} {op['path']} from its endpoint requirement. Return the schema only.\n\nRequirement:\n"+prompt_input
                rows.append({'text':chat(prompt,target),'kind':'response_schema'})
            break
    return rows

methods={'get','post','put','patch','delete','head','options','trace'}
ops=[]; title=str((api_spec.get('info') or {}).get('title') or 'REST API')
for path,item in (api_spec.get('paths') or {}).items():
    if isinstance(item,dict):
        for method,op in item.items():
            if method.lower() in methods and isinstance(op,dict): ops.append(make_op(title,method,path,item,op))
if len(ops)<20: raise RuntimeError(f'Only {len(ops)} API operations parsed.')
rng=random.Random(42); rng.shuffle(ops); cut=max(1,int(len(ops)*.9)); op_tr,op_ev=ops[:cut],ops[cut:]
api_tr=[x for op in op_tr for x in api_tasks(op)]; api_ev=[x for op in op_ev for x in api_tasks(op)]
rng.shuffle(api_ev); api_ev=api_ev[:MAX_API_EVAL]
if not api_tr or not api_ev: raise RuntimeError('OpenAPI train/validation task set is empty.')
print('SQL rows:',len(sql_tr),'train /',len(sql_ev),'eval; OpenAPI operations:',len(ops))
print('API tasks:',len(api_tr),'train /',len(api_ev),'eval')

In [ ]:
api_tr_ds=Dataset.from_list([{'text':r['text']} for r in api_tr]); api_ev_ds=Dataset.from_list([{'text':r['text']} for r in api_ev])
api_tr_ds=concatenate_datasets([api_tr_ds]*API_REPEAT)
train_text=concatenate_datasets([sql_tr.remove_columns([c for c in sql_tr.column_names if c!='text']),api_tr_ds]).shuffle(seed=42)
eval_text=concatenate_datasets([sql_ev.remove_columns([c for c in sql_ev.column_names if c!='text']),api_ev_ds]).shuffle(seed=43)
def tokenize(batch): return tokenizer(batch['text'],truncation=False)
train_all=train_text.map(tokenize,batched=True,remove_columns=['text']); eval_all=eval_text.map(tokenize,batched=True,remove_columns=['text'])
train_ds=train_all.filter(lambda r:len(r['input_ids'])<=MAX_LENGTH); eval_ds=eval_all.filter(lambda r:len(r['input_ids'])<=MAX_LENGTH)
if not len(train_ds) or not len(eval_ds): raise RuntimeError('No examples remain after length filtering.')
token_count=sum(len(x) for x in train_ds['input_ids'])
manifest={'run_id':RUN_ID,'specialist':'fullstack_api_backend_schema','base_model_id':BASE_MODEL_ID,'base_revision':BASE_REV,
 'sql_dataset_id':SQL_DATASET_ID,'sql_revision':SQL_REV,'sql_train_examples':len(sql_tr),'sql_eval_examples':len(sql_ev),
 'api_repo':API_REPO,'api_file':API_FILE,'api_commit':API_COMMIT,'api_sha256':API_SHA256,'api_operations':len(ops),
 'api_train_operations':len(op_tr),'api_eval_operations':len(op_ev),'api_train_tasks':len(api_tr),'api_eval_tasks':len(api_ev),
 'api_repeat_factor':API_REPEAT,'mixed_train_examples':len(train_ds),'mixed_eval_examples':len(eval_ds),'train_tokens':token_count,
 'max_length':MAX_LENGTH,'max_sql_train':MAX_SQL_TRAIN,'max_sql_eval':MAX_SQL_EVAL,'max_api_eval':MAX_API_EVAL,'target_modules':TARGET_MODULES,
 'lora_rank':RANK,'lora_alpha':ALPHA,'transformers_version':transformers.__version__,'torch_version':torch.__version__,
 'device_name':DEVICE,'started_utc':START_UTC,'save_steps':SAVE_STEPS,'eval_steps':EVAL_STEPS,'logging_steps':LOGGING_STEPS}
if prior:
    keys=['base_revision','sql_revision','api_commit','api_sha256','target_modules','lora_rank','lora_alpha','max_length','api_repeat_factor','max_sql_train','max_sql_eval','max_api_eval','save_steps','eval_steps','logging_steps']
    changed=[k for k in keys if prior.get(k)!=manifest.get(k)]
    if changed: raise RuntimeError(f'RUN_ID config differs for {changed}; choose a new RUN_ID.')
    manifest['started_utc']=prior.get('started_utc',START_UTC)
MANIFEST.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Mixed rows:',len(train_ds),'train /',len(eval_ds),'eval; train tokens:',token_count)
print('Filtered over',MAX_LENGTH,'tokens:',len(train_all)-len(train_ds),'train /',len(eval_all)-len(eval_ds),'eval')

## 2. Build the exact-base Q/MLP adapter

The trainable LoRA module names are checked after PEFT wrapping. K/V projections stay frozen; run the separate Bridge notebook to verify cache behavior and continuation.

In [ ]:
base=AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID,revision=BASE_REV,torch_dtype=torch.float16,device_map={'':0},low_cpu_mem_usage=True)
base.config.use_cache=False; base.gradient_checkpointing_enable(); base.enable_input_require_grads()
peft_config=LoraConfig(r=RANK,lora_alpha=ALPHA,lora_dropout=0.05,bias='none',target_modules=TARGET_MODULES,task_type=TaskType.CAUSAL_LM)
model=get_peft_model(base,peft_config); model.print_trainable_parameters()
trainable_params,total_params=model.get_nb_trainable_parameters()
actual_modules=sorted({n.split('.lora_')[0].split('.')[-1] for n,_ in model.named_parameters() if '.lora_' in n})
if set(actual_modules)-set(TARGET_MODULES) or {'k_proj','v_proj'}&set(actual_modules): raise RuntimeError(f'Unexpected LoRA targets: {actual_modules}')
print('Verified LoRA modules:',actual_modules)

## 3. Checkpoints, logs, and resume

Drive checkpoints include adapter/model weights, optimizer, scheduler, trainer state, and RNG state every 100 optimizer steps. Evaluation runs every 500 steps. The callback flushes JSONL records and prints an ETA. Source commits are pinned in `run_manifest.json`, and resume selects the latest complete checkpoint.

In [ ]:
PROGRESS=OUTPUT_DIR/'training_progress.jsonl'; BASELINE=OUTPUT_DIR/'baseline_eval.json'; SUMMARY=OUTPUT_DIR/'adapter_metadata_summary.json'
FINAL=OUTPUT_DIR/'final_adapter'; BRIDGE=ROOT/'fullstack_api_adapter_bridge_manifest.json'
class DriveCallback(TrainerCallback):
    def __init__(self,path):
        self.path=Path(path); self.started=time.monotonic(); self.train_start=None; self.start_step=0; self.prior=0.0
        if self.path.exists():
            for line in self.path.read_text(encoding='utf-8').splitlines():
                try: self.prior=max(self.prior,float(json.loads(line).get('cumulative_elapsed_seconds',0)))
                except (ValueError,TypeError): pass
    def write(self,row):
        with self.path.open('a',encoding='utf-8') as f: f.write(json.dumps(row,default=float)+'\n'); f.flush(); os.fsync(f.fileno())
    def on_train_begin(self,args,state,control,**kw): self.train_start=time.monotonic(); self.start_step=state.global_step
    def on_log(self,args,state,control,logs=None,**kw):
        elapsed=time.monotonic()-self.started; train_elapsed=time.monotonic()-self.train_start if self.train_start else None
        steps=max(0,state.global_step-self.start_step); eta=(state.max_steps-state.global_step)*train_elapsed/steps if train_elapsed and steps and state.max_steps>state.global_step else None
        self.write({'event':'log','utc':datetime.now(timezone.utc).isoformat(),'step':state.global_step,'max_steps':state.max_steps,
          'epoch':state.epoch,'session_elapsed_seconds':elapsed,'cumulative_elapsed_seconds':self.prior+elapsed,'training_elapsed_seconds':train_elapsed,
          'eta_seconds':eta,'peak_vram_gb':torch.cuda.max_memory_allocated()/(1024**3),'metrics':logs or {}})
        if eta is not None: print(f'[ETA] {state.global_step}/{state.max_steps}; {eta/3600:.2f} h at current rate')
    def on_save(self,args,state,control,**kw): self.write({'event':'checkpoint_saved','step':state.global_step,'path':str(Path(args.output_dir)/f'checkpoint-{state.global_step}')})
def latest_complete(directory):
    paths=sorted(Path(directory).glob('checkpoint-*'),key=lambda p:int(p.name.rsplit('-',1)[-1]) if p.name.rsplit('-',1)[-1].isdigit() else -1)
    for p in reversed(paths):
        names={x.name for x in p.iterdir()} if p.is_dir() else set()
        if 'trainer_state.json' in names and 'scheduler.pt' in names and any(n.startswith('adapter_model.') for n in names) and any(n in {'optimizer.pt','optimizer.safetensors'} for n in names) and any(n.startswith('rng_state') for n in names): return str(p)
    return None
checkpoint=latest_complete(OUTPUT_DIR)
if checkpoint: print('Resume from:',checkpoint)
else:
    found=get_last_checkpoint(str(OUTPUT_DIR))
    if found: print('Skipping incomplete checkpoint:',found)
training_args=TrainingArguments(output_dir=str(OUTPUT_DIR),run_name=RUN_ID,num_train_epochs=1,
 per_device_train_batch_size=1,per_device_eval_batch_size=2,gradient_accumulation_steps=8,gradient_checkpointing=True,fp16=True,
 learning_rate=2e-5,lr_scheduler_type='cosine',warmup_ratio=0.03,eval_strategy='steps',eval_steps=EVAL_STEPS,
 logging_strategy='steps',logging_steps=LOGGING_STEPS,logging_first_step=True,save_strategy='steps',save_steps=SAVE_STEPS,
 save_total_limit=3,save_only_model=False,load_best_model_at_end=False,logging_dir=str(LOG_DIR),report_to=['tensorboard'],
 include_tokens_per_second=True,include_num_input_tokens_seen=True,skip_memory_metrics=False,optim='adamw_torch',seed=42,data_seed=42,overwrite_output_dir=False)
trainer=Trainer(model=model,args=training_args,train_dataset=train_ds,eval_dataset=eval_ds,
 data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False),callbacks=[DriveCallback(PROGRESS)])

## 4. Train, save, and capture telemetry

Initial/final validation loss and perplexity, step losses, learning rate, target modules, parameter counts, adapter/checkpoint size, GPU, peak VRAM, throughput, and wall time are written to Drive. The final inference adapter and the larger resume checkpoints are separate artifacts.

In [ ]:
def atomic_json(path,value):
    path=Path(path); temp=path.with_suffix(path.suffix+'.tmp'); temp.write_text(json.dumps(value,indent=2,default=float),encoding='utf-8'); os.replace(temp,path)
def size_bytes(path):
    p=Path(path); return sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) if p.exists() else 0
if SUMMARY.exists() and FINAL.exists():
    print('Run already complete:'); print(SUMMARY.read_text(encoding='utf-8'))
else:
    if checkpoint and BASELINE.exists(): initial=json.loads(BASELINE.read_text(encoding='utf-8'))
    else:
        initial=trainer.evaluate(); initial['eval_perplexity']=math.exp(min(initial['eval_loss'],700)); atomic_json(BASELINE,initial)
    torch.cuda.reset_peak_memory_stats(); started=time.monotonic()
    try: result=trainer.train(resume_from_checkpoint=checkpoint if RESUME and checkpoint else None)
    except Exception as exc:
        atomic_json(OUTPUT_DIR/'last_failure.json',{'utc':datetime.now(timezone.utc).isoformat(),'error':repr(exc),'latest_complete_checkpoint':latest_complete(OUTPUT_DIR)}); raise
    wall=time.monotonic()-started
    model.save_pretrained(FINAL,safe_serialization=True); tokenizer.save_pretrained(FINAL)
    final=trainer.evaluate(); final['eval_perplexity']=math.exp(min(final['eval_loss'],700)); final['eval_loss_delta_from_baseline']=final['eval_loss']-initial['eval_loss']
    history=trainer.state.log_history; train_logs=[r for r in history if 'loss' in r and 'eval_loss' not in r]; eval_logs=[r for r in history if 'eval_loss' in r]
    for r in eval_logs: r['eval_perplexity']=math.exp(min(r['eval_loss'],700))
    atomic_json(OUTPUT_DIR/'training_history.json',{'initial_eval':initial,'final_eval':final,'trainer_log_history':history,'progress_jsonl':str(PROGRESS)})
    observed=sorted({n.split('.lora_')[0].split('.')[-1] for n,_ in model.named_parameters() if '.lora_' in n})
    param_summary={'trainable_parameters':int(trainable_params),'total_parameters':int(total_params),'trainable_percent':100*trainable_params/total_params,
      'target_modules_requested':TARGET_MODULES,'target_modules_observed':observed,'k_proj_or_v_proj_adapted':bool({'k_proj','v_proj'}&set(observed))}
    rows=[]
    if PROGRESS.exists():
        for line in PROGRESS.read_text(encoding='utf-8').splitlines():
            try: rows.append(json.loads(line))
            except ValueError: pass
    cp=latest_complete(OUTPUT_DIR); cp_size=size_bytes(cp) if cp else 0; final_size=size_bytes(FINAL)
    summary={'run_id':RUN_ID,'specialist':'fullstack_api_backend_schema','base_model_id':BASE_MODEL_ID,'base_revision':BASE_REV,
      'sql_dataset_id':SQL_DATASET_ID,'sql_revision':SQL_REV,'api_spec_repo':API_REPO,'api_spec_commit':API_COMMIT,'api_spec_sha256':API_SHA256,
      'sql_train_examples':len(sql_tr),'api_operations':len(ops),'api_train_tasks':len(api_tr),'api_repeat_factor':API_REPEAT,
      'mixed_train_examples':len(train_ds),'mixed_eval_examples':len(eval_ds),'initial_train_loss':train_logs[0]['loss'] if train_logs else None,
      'final_train_loss':train_logs[-1]['loss'] if train_logs else None,'initial_eval_loss':initial.get('eval_loss'),'initial_eval_perplexity':initial.get('eval_perplexity'),
      'final_eval_loss':final.get('eval_loss'),'final_eval_perplexity':final.get('eval_perplexity'),'best_logged_eval_loss':min((r['eval_loss'] for r in eval_logs),default=None),
      'global_step':trainer.state.global_step,'last_logged_learning_rate':next((r.get('learning_rate') for r in reversed(history) if 'learning_rate' in r),None),
      'parameters':param_summary,'hardware_compute':{'device_name':torch.cuda.get_device_name(0),'peak_vram_gb':torch.cuda.max_memory_allocated()/(1024**3),
      'training_session_wall_seconds':wall,'cumulative_logged_training_seconds':max([0.0]+[float(r.get('cumulative_elapsed_seconds',0)) for r in rows]),
      'train_samples_per_second':result.metrics.get('train_samples_per_second'),'train_tokens_per_second':result.metrics.get('train_tokens_per_second')},
      'latest_complete_checkpoint':cp,'checkpoint_size_mb':cp_size/(1024**2),'final_adapter_path':str(FINAL),'final_adapter_size_mb':final_size/(1024**2),
      'saved_utc':datetime.now(timezone.utc).isoformat()}
    atomic_json(SUMMARY,summary)
    atomic_json(BRIDGE,{'specialist':'fullstack_api_backend_schema','adapter_path':str(FINAL),'base_model_id':BASE_MODEL_ID,'base_revision':BASE_REV,
      'target_modules':observed,'sql_dataset_id':SQL_DATASET_ID,'sql_revision':SQL_REV,'api_spec_repo':API_REPO,'api_spec_commit':API_COMMIT,'run_id':RUN_ID,'summary_path':str(SUMMARY)})
    print(json.dumps(summary,indent=2,default=float)); print('Adapter:',FINAL); print('Bridge manifest:',BRIDGE)

## 5. Resume and use the adapter

After a Colab disconnect, reconnect the same Drive and rerun with the same `RUN_ID`. The notebook reuses the recorded model, SQL dataset, and OpenAPI commits, then resumes from the latest complete checkpoint. Choose a new `RUN_ID` for a fresh experiment.

TensorBoard:

```python
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/cgr_adapter_checkpoints/fullstack_api_qwen3_0.6b_q_mlp_r16_v1/tensorboard
```

The inference adapter is in `final_adapter/`; `checkpoint-N/` folders hold resume state. `fullstack_api_adapter_bridge_manifest.json` records the final adapter path and required base revision. Run the separate Bridge notebook to verify cache behavior and continuation.